In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, LassoCV, RidgeCV, ElasticNetCV
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
RS = 42

In [3]:
bf = fetch_openml('black_friday', version=1, as_frame=True, data_home='../data/raw/sklearn').frame
bf = bf.astype({c: 'object' for c in bf.select_dtypes('category').columns})
print(bf.shape)
bf.head()

(166821, 10)


,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
0,F,0-17,10,A,2,0,1,6,14,15200
1,M,46-50,7,B,2,1,1,8,17,19215
2,M,26-35,20,A,1,1,1,2,5,15665
3,F,51-55,9,A,1,0,5,8,14,5378
4,F,51-55,9,A,1,0,2,3,4,13055


In [4]:
print('Пропусков до:', bf.isna().sum().sum())
bf = bf.fillna(0)
print('Пропусков после:', bf.isna().sum().sum())

Пропусков до: 0
Пропусков после: 0


In [5]:
bf.dtypes

Gender                        object
Age                           object
Occupation                     int64
City_Category                 object
Stay_In_Current_City_Years    object
Marital_Status                 int64
Product_Category_1             int64
Product_Category_2             int64
Product_Category_3             int64
Purchase                       int64
dtype: object

In [6]:
print(bf['Gender'].unique())
bf['Gender'] = bf['Gender'].map({'M': 1, 'F': 0})

['F' 'M']


In [7]:
ages = sorted(bf['Age'].unique(), key=lambda s: int(s.split('-')[0].rstrip('+')))
print(ages)

def age_to_num(age):
    return ages.index(age)

bf['Age'] = bf['Age'].apply(age_to_num)
bf['Age'].value_counts().sort_index()

['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']


Age
0     4873
1    31316
2    66942
3    33285
4    13374
5    11166
6     5865
Name: count, dtype: int64

In [8]:
print(bf['City_Category'].unique())
bf = pd.get_dummies(bf, columns=['City_Category'], dtype=int)
bf.filter(like='City').head()

['A' 'B' 'C']


,Stay_In_Current_City_Years,City_Category_A,City_Category_B,City_Category_C
0,2,1,0,0
1,2,0,1,0
2,1,1,0,0
3,1,1,0,0
4,1,1,0,0


In [9]:
print(bf['Stay_In_Current_City_Years'].unique())
bf['Stay_In_Current_City_Years'] = bf['Stay_In_Current_City_Years'].replace('4+', '4').astype(int)
bf['Stay_In_Current_City_Years'].unique()

['2' '1' '4+' '0' '3']


array([2, 1, 4, 0, 3])

In [10]:
bf = bf.drop(columns=['User_ID', 'Product_ID'], errors='ignore')
print(bf.dtypes)

Gender                        int64
Age                           int64
Occupation                    int64
Stay_In_Current_City_Years    int64
Marital_Status                int64
Product_Category_1            int64
Product_Category_2            int64
Product_Category_3            int64
Purchase                      int64
City_Category_A               int64
City_Category_B               int64
City_Category_C               int64
dtype: object


In [14]:
X = bf.drop(columns='Purchase')
y = bf['Purchase']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RS)

scaler = StandardScaler().fit(X_train)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

lin = LinearRegression().fit(Xs_train, y_train)
pred = lin.predict(Xs_test)
print(f'RMSE = {mean_squared_error(y_test, pred) ** 0.5:.1f},  R2 = {r2_score(y_test, pred):.4f}')
pd.Series(lin.coef_, index=X.columns).sort_values()

RMSE = 4632.1,  R2 = 0.1745


Product_Category_1           -2113.673400
City_Category_A               -150.605323
City_Category_B                -75.616382
Marital_Status                 -24.622342
Stay_In_Current_City_Years      28.788643
Occupation                      39.384474
Product_Category_2             104.010003
Gender                         131.283334
Age                            194.200435
City_Category_C                216.944899
Product_Category_3             300.653910
dtype: float64

In [15]:
cv = KFold(5, shuffle=True, random_state=RS)
Xs = StandardScaler().fit_transform(X)
cv_r2 = cross_val_score(LinearRegression(), Xs, y, cv=cv, scoring='r2')
cv_rmse = -cross_val_score(LinearRegression(), Xs, y, cv=cv, scoring='neg_root_mean_squared_error')
print('R2 по фолдам:', cv_r2.round(4), ' среднее:', cv_r2.mean().round(4))
print('RMSE среднее:', cv_rmse.mean().round(1))

R2 по фолдам: [0.1729 0.1686 0.1681 0.1709 0.1667]  среднее: 0.1694
RMSE среднее: 4631.7


In [16]:
alphas = [0.1, 1, 10, 50, 100, 300, 1000]
lasso_w = pd.DataFrame({a: Lasso(alpha=a).fit(Xs_train, y_train).coef_ for a in alphas}, index=X.columns)
lasso_w.round(1)

,0.1,1.0,10.0,50.0,100.0,300.0,1000.0
Gender,131.2,130.5,123.4,91.3,47.1,0.0,0.0
Age,194.1,192.9,181.5,145.3,100.8,0.0,0.0
Occupation,39.3,38.7,32.2,2.6,0.0,0.0,0.0
Stay_In_Current_City_Years,28.7,27.8,19.4,0.0,0.0,0.0,0.0
Marital_Status,-24.5,-23.2,-10.3,-0.0,0.0,0.0,0.0
Product_Category_1,-2113.5,-2111.8,-2094.8,-2019.2,-1949.6,-1699.4,-1000.4
Product_Category_2,103.9,102.3,87.1,19.9,0.0,0.0,-0.0
Product_Category_3,300.6,300.2,296.3,278.6,226.9,0.0,0.0
City_Category_A,-264.5,-83.6,-77.7,-51.7,-18.5,-0.0,-0.0
City_Category_B,-205.4,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0


In [18]:
w100 = lasso_w[100]
selected = w100[w100 != 0].index.tolist()
print('Отобраны:', selected)
print('Отброшены:', w100[w100 == 0].index.tolist())
idx = [X.columns.get_loc(c) for c in selected]
lin_sel = LinearRegression().fit(Xs_train[:, idx], y_train)
print(f'R2 на test с отобранными признаками: {r2_score(y_test, lin_sel.predict(Xs_test[:, idx])):.4f}')

Отобраны: ['Gender', 'Age', 'Product_Category_1', 'Product_Category_3', 'City_Category_A', 'City_Category_C']
Отброшены: ['Occupation', 'Stay_In_Current_City_Years', 'Marital_Status', 'Product_Category_2', 'City_Category_B']
R2 на test с отобранными признаками: 0.1739


In [19]:
lasso_cv = LassoCV(alphas=np.logspace(-2, 3, 30), cv=cv).fit(Xs_train, y_train)
print(f'Оптимальный alpha (L1) = {lasso_cv.alpha_:.3f},  R2 на test = {lasso_cv.score(Xs_test, y_test):.4f}')

Оптимальный alpha (L1) = 0.022,  R2 на test = 0.1745


In [20]:
ridge_alphas = [0.1, 10, 1000, 1e4, 1e5, 1e6]
ridge_w = pd.DataFrame({a: Ridge(alpha=a).fit(Xs_train, y_train).coef_ for a in ridge_alphas}, index=X.columns)
ridge_w.round(1)

,0.1,10.0,1000.0,10000.0,100000.0,1000000.0
Gender,131.3,131.3,131.8,134.5,113.9,29.5
Age,194.2,194.2,191.8,173.0,88.8,14.8
Occupation,39.4,39.4,39.6,41.3,39.2,11.4
Stay_In_Current_City_Years,28.8,28.8,28.6,27.0,17.8,4.1
Marital_Status,-24.6,-24.6,-23.8,-17.7,0.9,1.7
Product_Category_1,-2113.7,-2113.4,-2089.3,-1897.2,-1039.2,-205.1
Product_Category_2,104.0,103.9,92.0,7.4,-175.7,-70.9
Product_Category_3,300.7,300.6,299.1,280.1,113.1,-3.6
City_Category_A,-150.6,-150.6,-150.0,-144.7,-102.7,-24.9
City_Category_B,-75.6,-75.6,-75.4,-73.0,-54.0,-14.2


In [21]:
ridge_cv = RidgeCV(alphas=np.logspace(-2, 6, 40), cv=cv).fit(Xs_train, y_train)
print(f'Оптимальный alpha (L2) = {ridge_cv.alpha_:.3f},  R2 на test = {ridge_cv.score(Xs_test, y_test):.4f}')

Оптимальный alpha (L2) = 49.239,  R2 на test = 0.1745


In [22]:
enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], alphas=np.logspace(-2, 2, 20), cv=cv, max_iter=10000).fit(Xs_train, y_train)
print(f'alpha = {enet.alpha_:.4f}, l1_ratio = {enet.l1_ratio_},  R2 на test = {enet.score(Xs_test, y_test):.4f}')
pd.DataFrame({'Linear': lin.coef_, 'Lasso': lasso_cv.coef_, 'Ridge': ridge_cv.coef_, 'ElasticNet': enet.coef_},
             index=X.columns).round(1)

alpha = 0.0162, l1_ratio = 1.0,  R2 на test = 0.1745


,Linear,Lasso,Ridge,ElasticNet
Gender,131.3,131.3,131.3,131.3
Age,194.2,194.2,194.1,194.2
Occupation,39.4,39.4,39.4,39.4
Stay_In_Current_City_Years,28.8,28.8,28.8,28.8
Marital_Status,-24.6,-24.6,-24.6,-24.6
Product_Category_1,-2113.7,-2113.6,-2112.5,-2113.6
Product_Category_2,104.0,104.0,103.4,104.0
Product_Category_3,300.7,300.6,300.6,300.6
City_Category_A,-150.6,-265.1,-150.6,-265.1
City_Category_B,-75.6,-206.0,-75.6,-206.1
